# Final-state radiation

Charged particles in the final state can radiate photons. EvtGen simulates this final-state radiation (FSR) for decay channels that carry the `FSR` (or legacy `PHOTOS`) flag in the decay file. {py:class}`.Generator` uses the QED shower of Vincia, which is part of Pythia8, when it is created with `fsr=True`. PHOTOS++ is not included.

The default decay table flags $J/\psi \to \mu^+\mu^-$ for FSR:

In [ ]:
from pyevtgen import DECAY_FILE

lines = DECAY_FILE.read_text().splitlines()
start = lines.index("Decay J/psi")
print("\n".join(lines[start : start + 3]))

Because a process can hold only one {py:class}`.Generator`, comparing FSR on and off requires two processes. The function below generates decays in a fresh Python process, with a user decay file that forces $J/\psi \to \mu^+\mu^-$ including the `FSR` flag. For each decay, it returns the dimuon invariant mass, the number of photons, and their total energy.

In [ ]:
import subprocess  # ruff: ignore[suspicious-subprocess-import]
import sys
from pathlib import Path
from tempfile import mkdtemp
from textwrap import dedent

import numpy as np

DIMUON_FILE = Path(mkdtemp()) / "dimuon.dec"
DIMUON_FILE.write_text("""
Decay J/psi
1.0 mu+ mu- FSR VLL;
Enddecay
End
""")


def generate_dimuons(*, fsr: bool, n: int = 5000) -> np.ndarray:
    """Return dimuon mass, photon count, and summed photon energy per event."""
    script = dedent(f"""
        import numpy as np
        from pyevtgen import Generator
        generator = Generator(fsr={fsr}, seed=0, user_decay_file={str(DIMUON_FILE)!r})
        events = generator.generate("J/psi", n={n})
        for event in events:
            names = [d.name for d in event.daughters]
            if "mu+" not in names or "mu-" not in names:
                continue
            muons = sum(np.array(d.p4) for d in event.daughters if d.name.startswith("mu"))
            mass = np.sqrt(muons[0] ** 2 - np.sum(muons[1:] ** 2))
            photons = [d.p4[0] for d in event.daughters if d.name == "gamma"]
            print("ROW", mass, len(photons), sum(photons))
    """)
    result = subprocess.run(  # ruff: ignore[subprocess-without-shell-equals-true]
        [sys.executable, "-c", script], capture_output=True, check=True, text=True
    )
    lines = result.stdout.splitlines()
    rows = [line.split()[1:] for line in lines if line.startswith("ROW")]
    return np.array(rows, dtype=float)

In [ ]:
samples = {fsr: generate_dimuons(fsr=fsr, n=20_000) for fsr in [False, True]}
for fsr, rows in samples.items():
    fraction = np.mean(rows[:, 1] > 0)
    print(f"fsr={fsr}: {len(rows)} decays, {fraction:.1%} with photons")

The radiated energy is missing from the muon pair, so the dimuon invariant mass develops a radiative tail below the $J/\psi$ mass. The photon energy spectrum is steeply falling, as expected for soft radiation:

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats

set_matplotlib_formats("svg")
fig, axes = plt.subplots(ncols=2, figsize=(9, 3.2))
for fsr, rows in samples.items():
    axes[0].hist(
        rows[:, 0], bins=80, range=(2.6, 3.12), histtype="step", label=f"fsr={fsr}"
    )
radiated = samples[True][:, 2]
axes[1].hist(radiated[radiated > 0], bins=60)
axes[0].set_xlabel(r"$m(\mu^+\mu^-)$ [GeV]")
axes[0].set_yscale("log")
axes[0].legend()
axes[1].set_xlabel(r"$E_\gamma$ [GeV]")
axes[1].set_yscale("log")
fig.tight_layout()
plt.show()